In [ ]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files={'train': 'cleaned_mixed_data_no_punctuation1.csv', "test":""}, split='train')
dataset = dataset.train_test_split(test_size=0.2)


Generating train split: 0 examples [00:00, ? examples/s]

In [19]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert/distilbert-base-uncased")

# Preprocess function
def preprocess(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=128)

# Apply to dataset
encoded_dataset = dataset.map(preprocess, batched=True)


Map:   0%|          | 0/26432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6609 [00:00<?, ? examples/s]

In [20]:
encoded_dataset = encoded_dataset.rename_column("Label", "labels")

In [21]:
encoded_dataset

DatasetDict({
    train: Dataset({
        features: ['Text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 26432
    })
    test: Dataset({
        features: ['Text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6609
    })
})

In [22]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
encoded_dataset.keys()

dict_keys(['train', 'test'])

In [24]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./distilbert-speaksense",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\Rohit Francis\AppData\Local\Temp\ipykernel_8480\583560298.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

In [4]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.cuda.get_device_name(0))  # Print GPU name


True
NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./distilbert-speaksense/checkpoint-1625", tokenizer="./distilbert-speaksense/checkpoint-1625")
classifier("I gotta have my lunch and after that go to work again, what do you think I should do after taht")
# Output: [{'label': 'LABEL_1', 'score': 0.9876}]


Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.9986746311187744}]

In [30]:
print(classifier("I gotta have my lunch and after that go to work again, what do you think I should do after taht"))
print(classifier("I was planning to make a really good martini"))
print(classifier("I was planning to make a really good martini, do you like martini?"))
print(classifier("So I'm going to take an event on...on quantum computing. I only know little about..."))
print(classifier("about it by the little I mean...really little like I only made some...A small game."))

[{'label': 'LABEL_0', 'score': 0.9986746311187744}]
[{'label': 'LABEL_1', 'score': 0.9766422510147095}]
[{'label': 'LABEL_0', 'score': 0.9982840418815613}]
[{'label': 'LABEL_0', 'score': 0.8667097687721252}]
[{'label': 'LABEL_1', 'score': 0.9889171123504639}]


In [31]:
print(classifier("that is not at all working efficient remain "))
print(classifier("That is not that much of a cool thing.That is not that much of a cool thing to do, man.Anyways, let us move forward to something great."))
print(classifier("all that stuff.So, if I say so, you will actually"))
print(classifier("about a really cool topic which is AI andAI andRobotics and all that stuff. So, um..."))
print(classifier("about a really cool topic which is AI andAI and"))
print(classifier("about a really cool topic which is AI and"))
print(classifier("that is how we actually got little here.Although the model is not that much good at it, what it.Should I have done or what it should be doing"))
print(classifier("done or what it should be doing but we are actually making it better."))
print(classifier("and make examples of it and that is how we actually got."))
print(classifier("of real world data and make examples of it."))
print(classifier("in order to train the model we actually had.To make use of re-"))
print(classifier("drag with robots and out."))
print(classifier("I mean currently factors are caused by blood pressure.This next generation technology which can actually redefine.We will be redefined how we interact with robots and all."))
print(classifier("I mean currently factors are caused by blood pressure.This next generation technology which can actually redefine.We will be redefined how we interact with robots and all."))
print(classifier("use our technology which is big sense."))
print(classifier("can actually use would be to use."))
print(classifier("So I think the best technology that we can actually use would be."))
print(classifier("technology man we don't actually need it like we need something which is morenatural for robots to understand.So I think the best"))
print(classifier("Who needs this vague word technology man We don't actually need"))

[{'label': 'LABEL_0', 'score': 0.8502442836761475}]
[{'label': 'LABEL_0', 'score': 0.9962385892868042}]
[{'label': 'LABEL_0', 'score': 0.9986893534660339}]
[{'label': 'LABEL_0', 'score': 0.6556369066238403}]
[{'label': 'LABEL_1', 'score': 0.9825795888900757}]
[{'label': 'LABEL_1', 'score': 0.9813022017478943}]
[{'label': 'LABEL_0', 'score': 0.9023415446281433}]
[{'label': 'LABEL_1', 'score': 0.9565541744232178}]
[{'label': 'LABEL_0', 'score': 0.9974880218505859}]
[{'label': 'LABEL_0', 'score': 0.9976258873939514}]
[{'label': 'LABEL_0', 'score': 0.9839043617248535}]
[{'label': 'LABEL_0', 'score': 0.8302580118179321}]
[{'label': 'LABEL_0', 'score': 0.7138493061065674}]
[{'label': 'LABEL_0', 'score': 0.7138493061065674}]
[{'label': 'LABEL_0', 'score': 0.9817355275154114}]
[{'label': 'LABEL_0', 'score': 0.9973914623260498}]
[{'label': 'LABEL_0', 'score': 0.9812461733818054}]
[{'label': 'LABEL_0', 'score': 0.9666682481765747}]
[{'label': 'LABEL_0', 'score': 0.9789122939109802}]


In [ ]:

print(classifier("...So ... But to say ... My true thinking what to say ...Do you have any opinion on any food that ..... I should try ... It seems like you're trying to convey ..."))













In [ ]:
done or what it should be doing but we are actually making it better.Making it better.Mainstuff is it is not able to recognize how to.
done or what it should be doing but we are actually making it better.Making it better.
in order to train the model we actually had.
